In [1]:
!pip install rouge-score bert-score accelerate datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.7 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login

HF_TOKEN = "your_token"
login(HF_TOKEN)

In [3]:
import json
import random
from pathlib import Path
import torch
from datasets import load_dataset
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [ ]:
def load_gazeta_test_split(num_samples=500, seed=42):
    """Загрузка датасета IlyaGusev/gazeta (русские новости)"""
    # Загружаем тестовую часть
    dataset = load_dataset("IlyaGusev/gazeta", split="test")
    
    random.seed(seed)
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))
    
    samples = []
    for idx in indices:
        item = dataset[idx]
        samples.append({
            'text': item['text'],        # Оригинальный текст статьи
            'summary': item['summary']    # Краткое содержание
        })
    
    return samples

# Альтернативный датасет
def load_summarus_test_split(num_samples=200, seed=42):
    """Загрузка датасета RussianNLP/summarus"""
    try:
        dataset = load_dataset("RussianNLP/summarus", split="test")
    except:
        dataset = load_dataset("RussianNLP/summarus", split="train")
    
    random.seed(seed)
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))
    
    samples = []
    for idx in indices:
        item = dataset[idx]
        samples.append({
            'text': item['text'],
            'summary': item['summary']
        })
    
    return samples

In [5]:
def compute_rouge(references, predictions):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for ref, pred in zip(references, predictions):
        result = scorer.score(ref, pred)
        for key in scores:
            scores[key].append(result[key].fmeasure)
    return {key: sum(v) / len(v) if v else 0.0 for key, v in scores.items()}

def compute_bertscore(references, predictions, batch_size=32):
    device = 0 if torch.cuda.is_available() else -1
    P, R, F1 = bert_score_fn(
        predictions, references, lang='ru',
        batch_size=batch_size, device=device, verbose=False
    )
    return {
        'bertscore_precision': P.mean().item(),
        'bertscore_recall': R.mean().item(),
        'bertscore_f1': F1.mean().item()
    }

def compute_all_metrics(references, predictions, batch_size=32):
    print("Computing ROUGE...")
    rouge = compute_rouge(references, predictions)
    print("Computing BERTScore...")
    bert = compute_bertscore(references, predictions, batch_size)
    return {**rouge, **bert}

In [6]:
class SummarizationPipeline:
    def __init__(self, model_name, device="cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"Loading {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name, dtype=dtype
        )
        self.model.to(self.device)
        self.model.eval()
        print(f"Model loaded on {self.device}")
    
    def generate(self, text, max_input_tokens=450, max_output_tokens=300,
                 num_beams=4, repetition_penalty=2.5):
        input_text = f"summarize: {text}"
        inputs = self.tokenizer(
            input_text, return_tensors="pt",
            max_length=max_input_tokens, truncation=True
        ).to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs, max_new_tokens=max_output_tokens,
                num_beams=num_beams, early_stopping=True,
                repetition_penalty=repetition_penalty,
                no_repeat_ngram_size=3, length_penalty=1.0
            )
        return self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    def generate_batch(self, texts, batch_size=8, **kwargs):
        predictions = []
        for i in tqdm(range(0, len(texts), batch_size), desc="Generating"):
            batch = texts[i:i + batch_size]
            for text in batch:
                predictions.append(self.generate(text, **kwargs))
        return predictions

In [7]:
MODEL_NAME = "cointegrated/rut5-base-absum"
NUM_SAMPLES = 500
BATCH_SIZE = 8

In [8]:
# Загружаем модель
pipeline = SummarizationPipeline(MODEL_NAME)

# Загружаем датасет Gazeta
samples = load_gazeta_test_split(num_samples=NUM_SAMPLES)
print(f"Loaded {len(samples)} samples from IlyaGusev/gazeta")
print(f"\nExample text: {samples[0]['text'][:200]}...")
print(f"Example summary: {samples[0]['summary']}")

Loading cointegrated/rut5-base-absum...


config.json:   0%|          | 0.00/753 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/828k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

default/test/0000.parquet:   0%|          | 0.00/30.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60964 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6369 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6793 [00:00<?, ? examples/s]

Loaded 500 samples from IlyaGusev/gazeta

Example text: Участковый уполномоченный полиции из поселка Провидения на Чукотке незаконно сажал людей в самодельную металлическую клетку и избивал задержанных. Об этом сообщает Следственное управление СК России по...
Example summary: Участковый в поселке Янракыннот на Чукотке подозревается в незаконном удержании людей в железной клетке. По данным местного СК, полицейский запирал там девушку, женщину и молодого человека.


In [9]:
texts = [s['text'] for s in samples]
references = [s['summary'] for s in samples]

predictions = pipeline.generate_batch(
    texts, batch_size=BATCH_SIZE,
    max_input_tokens=450, max_output_tokens=300,
    num_beams=4, repetition_penalty=2.5
)

Generating: 100%|██████████| 63/63 [09:55<00:00,  9.45s/it]


In [10]:
metrics = compute_all_metrics(references, predictions, batch_size=32)

print("\n=== METRICS ===")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

Computing ROUGE...
Computing BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== METRICS ===
rouge1: 0.1756
rouge2: 0.0678
rougeL: 0.1740
bertscore_precision: 0.7431
bertscore_recall: 0.6544
bertscore_f1: 0.6954


In [11]:
print("\n=== EXAMPLES ===")
for i in range(5):
    print(f"\n--- Sample {i+1} ---")
    print(f"Reference: {references[i]}")
    print(f"Prediction: {predictions[i]}")


=== EXAMPLES ===

--- Sample 1 ---
Reference: Участковый в поселке Янракыннот на Чукотке подозревается в незаконном удержании людей в железной клетке. По данным местного СК, полицейский запирал там девушку, женщину и молодого человека.
Prediction: На Чукотском автономном округе находилась самодельная металлическая клетка.

--- Sample 2 ---
Reference: Бабье лето еще не кончилось – тепло сохранится в европейской части России до среды, заявили в центре «Фобос». Ветер ожидается в основном южный, несильный, а температура днем будет составлять от +16 до +22°C. На выходных похолодает, но погода будет солнечной и сухой.
Prediction: В центральной части европейской России продержатся летнее тепло до среды, сообщил ведущий специалист центра «Фобос» Евгений Тишковец

--- Sample 3 ---
Reference: Новый российский боевой железнодорожный ракетный комплекс «Баргузин» может представлять серьезную угрозу для США, считают китайские журналисты. Они отметили, что, несмотря на большое количество преимуществ

In [12]:
results = {
    'model': MODEL_NAME,
    'num_samples': NUM_SAMPLES,
    'metrics': metrics,
    'examples': [
        {'text': texts[i][:500], 'reference': references[i], 'prediction': predictions[i]}
        for i in range(10)
    ]
}

Path('/kaggle/working').mkdir(exist_ok=True)
with open('/kaggle/working/baseline.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ Saved to /kaggle/working/baseline.json")

✅ Saved to /kaggle/working/baseline.json
